In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

matches = list(Path(".").rglob("california_housing.csv"))
if not matches:
    raise FileNotFoundError("Extract the practical datasets ZIP beside this notebook.")
df = pd.read_csv(matches[0])
X = df[["MedInc"]]
y = df["MedHouseVal"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
degrees = [1, 2, 3, 5, 8]
results = []
models = {}

In [ ]:
for degree in degrees:
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        StandardScaler(),
        LinearRegression()
    )
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    train_mse = mean_squared_error(y_train, train_pred)
    test_mse = mean_squared_error(y_test, test_pred)
    results.append([degree, train_mse, test_mse, r2_score(y_test, test_pred)])
    models[degree] = model

results_df = pd.DataFrame(results, columns=["Degree", "Train MSE", "Test MSE", "Test R2"])
display(results_df.round(4))

In [ ]:
plt.figure(figsize=(8, 5))
x_line = np.linspace(X["MedInc"].min(), X["MedInc"].max(), 300).reshape(-1, 1)
sample = X.sample(min(1500, len(X)), random_state=42)
plt.scatter(sample["MedInc"], y.loc[sample.index], alpha=0.15, s=12, label="Data")
for degree in [1, 2, 3, 5]:
    plt.plot(x_line.ravel(), models[degree].predict(x_line), label=f"Degree {degree}")
plt.xlabel("Median Income")
plt.ylabel("Median House Value")
plt.title("Polynomial Regression with Different Degrees")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(results_df["Degree"], results_df["Train MSE"], marker="o", label="Train MSE")
plt.plot(results_df["Degree"], results_df["Test MSE"], marker="o", label="Test MSE")
plt.xlabel("Polynomial Degree")
plt.ylabel("Mean Squared Error")
plt.title("Model Complexity and Bias-Variance Trade-off")
plt.xticks(degrees)
plt.legend()
plt.grid(alpha=0.25)
plt.show()
print("Practical 4 Summary:")
print("Polynomial models of degrees 1, 2, 3, 5 and 8 were compared using median income as the input.")
print("Train/test errors illustrate how increasing model complexity can reduce bias but increase variance.")